# ProPpilot Dataset — LangGraph RAG Ingestion Pipeline

This notebook builds a **retrieval-ready FAISS vector store** from `proppilot_dataset.csv` using a **LangGraph** ingestion pipeline.

**Pipeline (a LangGraph `StateGraph`):**

```
START -> load_csv -> split_documents -> embed_and_store -> END
```

**Outputs (saved to `./faiss_index/`):**
- `index.faiss` — the FAISS vector index
- `index.pkl` — the pickled docstore + index-to-docstore-id mapping

These two files are exactly what `FAISS.save_local(...)` produces, and can later be reloaded with `FAISS.load_local(...)` to build a retrieval / RAG chain or a query-time LangGraph.

> **Note:** This notebook only handles **ingestion** (building the vector store). A separate notebook/graph can load these files and add retrieval + generation nodes for full RAG Q&A.

**Assumptions made (edit the CONFIG cell below if these don't match your needs):**
- `proppilot_dataset.csv` is already present in the Colab working directory (as shown in your file browser).
- Embeddings use a free, local HuggingFace model (`sentence-transformers/all-MiniLM-L6-v2`) so no API key is required. Swap in `OpenAIEmbeddings` if you'd rather use OpenAI (see commented-out code in the imports cell).
- Every column in the CSV is treated as text via `CSVLoader` (each row becomes one `Document`, with `column: value` pairs as content). If you want only specific columns embedded, change the `CSVLoader` args.


## 1. Install dependencies

In [ ]:
%%capture
!pip install -q -U langgraph langchain langchain-community langchain-huggingface langchain-text-splitters sentence-transformers faiss-cpu pandas


## 2. Imports

In [ ]:
import os
from typing import List
from typing_extensions import TypedDict

from langchain_core.documents import Document
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph, START, END

# --- Optional: use OpenAI embeddings instead of local HuggingFace embeddings ---
# import os
# os.environ["OPENAI_API_KEY"] = "sk-..."
# from langchain_openai import OpenAIEmbeddings
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


## 3. Configuration

In [ ]:
CSV_PATH = "proppilot_dataset.csv"      # path to the uploaded CSV (already in Colab per the file browser)
OUTPUT_DIR = "faiss_index"              # folder where index.faiss / index.pkl will be saved
INDEX_NAME = "index"                    # base filename -> index.faiss, index.pkl

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150


## 4. Define the LangGraph state

LangGraph pipelines pass a shared, typed **state** object between nodes. Each node reads what it needs from the
state and returns a partial dict of updates, which LangGraph merges back into the state automatically.

In [ ]:
class IngestionState(TypedDict):
    csv_path: str
    output_dir: str
    index_name: str
    documents: List[Document]
    chunks: List[Document]
    num_documents: int
    num_chunks: int


## 5. Define the graph nodes

In [ ]:
def load_csv_node(state: IngestionState) -> dict:
    """Load the CSV file into LangChain Document objects (one Document per row)."""
    loader = CSVLoader(file_path=state["csv_path"], encoding="utf-8")
    documents = loader.load()
    print(f"[load_csv] Loaded {len(documents)} rows from '{state['csv_path']}'")
    return {"documents": documents, "num_documents": len(documents)}


def split_documents_node(state: IngestionState) -> dict:
    """Split documents into smaller overlapping chunks for better retrieval quality."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    chunks = splitter.split_documents(state["documents"])
    print(f"[split_documents] Split {state['num_documents']} documents into {len(chunks)} chunks")
    return {"chunks": chunks, "num_chunks": len(chunks)}


def embed_and_store_node(state: IngestionState) -> dict:
    """Embed the chunks and persist a FAISS index (index.faiss + index.pkl) to disk."""
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

    vectorstore = FAISS.from_documents(state["chunks"], embeddings)

    os.makedirs(state["output_dir"], exist_ok=True)
    vectorstore.save_local(state["output_dir"], index_name=state["index_name"])

    saved_files = os.listdir(state["output_dir"])
    print(f"[embed_and_store] Saved FAISS vector store to '{state['output_dir']}/': {saved_files}")
    return {}


## 6. Build and compile the LangGraph pipeline

In [ ]:
graph_builder = StateGraph(IngestionState)

graph_builder.add_node("load_csv", load_csv_node)
graph_builder.add_node("split_documents", split_documents_node)
graph_builder.add_node("embed_and_store", embed_and_store_node)

graph_builder.add_edge(START, "load_csv")
graph_builder.add_edge("load_csv", "split_documents")
graph_builder.add_edge("split_documents", "embed_and_store")
graph_builder.add_edge("embed_and_store", END)

ingestion_graph = graph_builder.compile()


In [ ]:
# Optional: visualize the compiled graph structure
try:
    from IPython.display import Image, display
    display(Image(ingestion_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph visualization skipped (needs internet access for mermaid rendering):", e)
    print(ingestion_graph.get_graph().draw_ascii())


## 7. Run the pipeline

In [ ]:
initial_state: IngestionState = {
    "csv_path": CSV_PATH,
    "output_dir": OUTPUT_DIR,
    "index_name": INDEX_NAME,
    "documents": [],
    "chunks": [],
    "num_documents": 0,
    "num_chunks": 0,
}

final_state = ingestion_graph.invoke(initial_state)

print("\nDone. Final state summary:")
print(f"  documents: {final_state['num_documents']}")
print(f"  chunks:    {final_state['num_chunks']}")


## 8. Verify the output files

In [ ]:
output_files = os.listdir(OUTPUT_DIR)
print(f"Files in '{OUTPUT_DIR}/':")
for f in output_files:
    path = os.path.join(OUTPUT_DIR, f)
    size_kb = os.path.getsize(path) / 1024
    print(f"  - {f} ({size_kb:.1f} KB)")

assert f"{INDEX_NAME}.faiss" in output_files, "index.faiss not found!"
assert f"{INDEX_NAME}.pkl" in output_files, "index.pkl not found!"
print("\nFAISS index and pkl file generated successfully.")


## 9. (Optional) Quick sanity check — reload and query

This confirms the saved files are valid and retrieval works, before you build a full RAG/query graph on top of them.

In [ ]:
reload_embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

reloaded_vectorstore = FAISS.load_local(
    OUTPUT_DIR,
    reload_embeddings,
    index_name=INDEX_NAME,
    allow_dangerous_deserialization=True,  # safe here since we generated this pkl file ourselves
)

query = "What is this dataset about?"  # <-- try a real question relevant to your data
results = reloaded_vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(doc.page_content[:300])
    print()


## Next steps

To turn this into a full **RAG** system, build a second LangGraph `StateGraph` (a query-time graph) with nodes like:

- `retrieve` — `FAISS.load_local(...)` the saved index and call `.similarity_search()` / `.as_retriever()`
- `generate` — pass retrieved chunks + the user's question to a chat model (e.g. `ChatOpenAI`, `ChatAnthropic`) to produce the final answer

That graph would take a `question` in its state, route through `retrieve -> generate -> END`, and return the answer — reusing the `index.faiss` / `index.pkl` files created here.